In [5]:
# Imports
import math
import time
from typing import Any

import numpy as np
import pandas as pd

from hamover import HamoverSearch


def run_hamover_case(
    name: str,
    N: int,
    target: int,
    approach: str,
    backend: str = "none",
    **setup_kwargs: Any,
) -> dict[str, Any]:
    """Run one HamoverSearch configuration and return a compact record."""
    search = HamoverSearch(N=N, target=target)

    t0 = time.perf_counter()
    result = search.setup(approach=approach, backend=backend, **setup_kwargs).solve()
    wall_time_s = time.perf_counter() - t0

    diagnostics = dict(result.diagnostics)
    sim_method = diagnostics.get("simulation_method")
    n_segments = int(setup_kwargs.get("n_segments", 1))

    if sim_method == "qdrift":
        diagnostics["algorithmic_steps_estimate"] = n_segments * int(
            diagnostics.get("num_qdrift", setup_kwargs.get("num_qdrift", 20))
        )
    elif sim_method == "suzuki_trotter":
        diagnostics["algorithmic_steps_estimate"] = n_segments * int(
            diagnostics.get("trotter_reps", setup_kwargs.get("trotter_reps", 10))
        )

    return {
        "name": name,
        "N": N,
        "target": target,
        "algorithm": "hamiltonian_evolution",
        "approach": result.approach,
        "backend": result.backend,
        "probability": float(result.probability),
        "found": bool(result.target_found),
        "model_runtime": float(result.runtime),
        "wall_time_s": float(wall_time_s),
        "scaling_exponent": result.scaling_exponent,
        "diagnostics": diagnostics,
        "setup_options": dict(setup_kwargs),
        "result": result,
    }

### Hamover vs Grover (Oracle + Diffuser) on Classiq

This notebook compares two search styles on the same target and `N`:

1. **Hamiltonian evolution (Hamover)** through the existing Classiq backend (`qdrift`/`suzuki_trotter`).
2. **Gate-model Grover** with explicit `grover_oracle` and `grover_diffusion` functions.

The comparison includes richer metrics beyond depth/runtime:
- success probability,
- leakage and valid-shot rates,
- CX count and depth-normalized efficiency,
- wall-clock execution time,
- algorithmic effort (`oracle_calls` for Grover, estimated simulation steps for Hamover),
- absolute error vs each method's reference probability.

In [6]:
def _decode_classiq_state_to_index(raw_state: Any, n_qubits: int) -> int:
    """Decode a Classiq parsed state value into an integer basis index."""
    if isinstance(raw_state, (int, np.integer)):
        return int(raw_state)

    if isinstance(raw_state, (list, tuple)):
        bitstring = "".join(str(int(b)) for b in reversed(raw_state)).zfill(n_qubits)
        return int(bitstring, 2)

    if isinstance(raw_state, dict):
        if not raw_state:
            raise ValueError("Empty parsed state dictionary")
        return _decode_classiq_state_to_index(next(iter(raw_state.values())), n_qubits)

    raise TypeError(f"Unsupported parsed state type: {type(raw_state)}")


def run_classiq_grover_case(
    name: str,
    N: int,
    target: int,
    shots: int = 1000,
    grover_reps: int | None = None,
) -> dict[str, Any]:
    """Run Grover search with explicit oracle and diffusion using Classiq."""
    if N < 2:
        raise ValueError("N must be >= 2")
    if not (0 <= target < N):
        raise ValueError(f"target must be in [0, {N-1}]")
    if N & (N - 1):
        raise ValueError("Grover path expects N to be a power of two (2^n).")

    from classiq import (
        Const,
        Output,
        QArray,
        QBit,
        allocate,
        create_model,
        execute,
        grover_diffuser,
        hadamard_transform,
        phase_oracle,
        qfunc,
        qperm,
        set_quantum_program_execution_preferences,
        synthesize,
    )
    from classiq.execution import ExecutionPreferences
    from classiq.qmod.symbolic import logical_and

    n_qubits = int(math.log2(N))
    if grover_reps is None:
        grover_reps = max(1, int(round((math.pi / 4.0) * math.sqrt(N))))
    grover_reps = int(grover_reps)

    target_bits_lsb = f"{target:0{n_qubits}b}"[::-1]

    def _logical_and_all(terms: list[Any]) -> Any:
        """Classiq logical_and is binary in some versions; fold pairwise."""
        out = terms[0]
        for term in terms[1:]:
            out = logical_and(out, term)
        return out

    @qperm
    def is_target(x: Const[QArray[QBit]], indicator: QBit) -> None:
        checks = [x[i] == int(bit) for i, bit in enumerate(target_bits_lsb)]
        indicator ^= _logical_and_all(checks)

    @qfunc
    def grover_oracle(x: QArray[QBit]) -> None:
        phase_oracle(is_target, x)

    @qfunc
    def grover_diffusion(x: QArray[QBit]) -> None:
        grover_diffuser(hadamard_transform, x)

    @qfunc
    def grover_step(x: QArray[QBit]) -> None:
        grover_oracle(x)
        grover_diffusion(x)

    @qfunc
    def main(x: Output[QArray[QBit]]) -> None:
        allocate(n_qubits, x)
        hadamard_transform(x)
        for _ in range(grover_reps):
            grover_step(x)

    t0 = time.perf_counter()
    qprog = synthesize(create_model(main))
    qprog = set_quantum_program_execution_preferences(
        qprog, ExecutionPreferences(num_shots=int(shots))
    )
    results = execute(qprog).result()
    wall_time_s = time.perf_counter() - t0

    found = 0
    not_found = 0
    leakage = 0

    parsed_counts = results[0].value.parsed_counts
    for parsed_state in parsed_counts:
        state_dict = parsed_state.state
        raw_state = state_dict.get("x", next(iter(state_dict.values())))
        idx = _decode_classiq_state_to_index(raw_state, n_qubits)
        count = int(parsed_state.shots)

        if idx == target:
            found += count
        elif 0 <= idx < N:
            not_found += count
        else:
            leakage += count

    valid = found + not_found
    success_probability = float(found / valid) if valid > 0 else 0.0

    circuit_depth = 0
    cx_count = 0
    try:
        transpiled = qprog.transpiled_circuit
        depth_obj = getattr(transpiled, "depth", 0)
        circuit_depth = int(depth_obj() if callable(depth_obj) else depth_obj)

        count_ops = getattr(transpiled, "count_ops", {})
        count_ops = count_ops() if callable(count_ops) else count_ops
        if isinstance(count_ops, dict):
            cx_count = int(count_ops.get("cx", 0))
    except Exception:
        pass

    theta = math.asin(1.0 / math.sqrt(N))
    ideal_probability = float(math.sin((2 * grover_reps + 1) * theta) ** 2)

    return {
        "name": name,
        "N": N,
        "target": target,
        "algorithm": "grover_oracle_diffuser",
        "approach": "grover_oracle_diffuser",
        "backend": "classiq",
        "probability": success_probability,
        "found": bool(found > not_found),
        "model_runtime": float(grover_reps),
        "wall_time_s": float(wall_time_s),
        "scaling_exponent": 0.5,
        "diagnostics": {
            "backend_found_count": int(found),
            "backend_not_found_count": int(not_found),
            "backend_leakage_count": int(leakage),
            "circuit_depth": int(circuit_depth),
            "cx_count": int(cx_count),
            "oracle_calls": int(grover_reps),
            "diffuser_calls": int(grover_reps),
            "ideal_grover_probability": ideal_probability,
        },
        "setup_options": {
            "shots": int(shots),
            "grover_reps": int(grover_reps),
        },
        "result": None,
    }


def run_case(
    name: str,
    N: int,
    target: int,
    mode: str = "hamover",
    approach: str = "adiabatic",
    backend: str = "classiq",
    **kwargs: Any,
) -> dict[str, Any]:
    """Unified runner for hamover Hamiltonian evolution or Grover gate search."""
    mode_key = mode.lower().strip()
    if mode_key == "hamover":
        return run_hamover_case(
            name=name,
            N=N,
            target=target,
            approach=approach,
            backend=backend,
            **kwargs,
        )
    if mode_key in {"grover", "grover_oracle_diffuser"}:
        return run_classiq_grover_case(
            name=name,
            N=N,
            target=target,
            shots=int(kwargs.get("shots", 1000)),
            grover_reps=kwargs.get("grover_reps"),
        )
    raise ValueError("mode must be 'hamover' or 'grover_oracle_diffuser'")


def _safe_div(num: float, den: float) -> float:
    return float(num / den) if den else np.nan


def _record_metrics(record: dict[str, Any]) -> dict[str, Any]:
    d = record.get("diagnostics", {})
    found = int(d.get("backend_found_count", 0))
    not_found = int(d.get("backend_not_found_count", 0))
    leakage = int(d.get("backend_leakage_count", 0))
    shots = found + not_found + leakage
    valid = found + not_found

    depth = int(d.get("circuit_depth", 0) or 0)
    cx_count = int(d.get("cx_count", 0) or 0)
    prob = float(record["probability"])
    wall = float(record.get("wall_time_s", np.nan))

    reference = d.get("reference_probability_final", d.get("ideal_grover_probability"))
    abs_ref_error = np.nan if reference is None else abs(prob - float(reference))

    return {
        "name": record["name"],
        "N": int(record["N"]),
        "target": int(record["target"]),
        "algorithm": record["algorithm"],
        "approach": record["approach"],
        "backend": record["backend"],
        "success_probability": prob,
        "valid_shot_rate": _safe_div(valid, shots),
        "leakage_rate": _safe_div(leakage, shots),
        "found_shots": found,
        "not_found_shots": not_found,
        "leakage_shots": leakage,
        "circuit_depth": depth,
        "cx_count": cx_count,
        "success_per_100_cx": _safe_div(100.0 * prob, max(cx_count, 1)),
        "success_per_100_depth": _safe_div(100.0 * prob, max(depth, 1)),
        "model_runtime": float(record.get("model_runtime", np.nan)),
        "wall_time_s": wall,
        "probability_per_wall_second": _safe_div(prob, wall),
        "algorithmic_steps": d.get("oracle_calls", d.get("algorithmic_steps_estimate", np.nan)),
        "reference_error_abs": abs_ref_error,
        "scaling_exponent": record.get("scaling_exponent"),
    }


def compare_records(records: list[dict[str, Any]]) -> pd.DataFrame:
    rows = [_record_metrics(r) for r in records]
    return (
        pd.DataFrame(rows)
        .sort_values(["N", "algorithm", "name"])
        .reset_index(drop=True)
    )

In [7]:
# Single-N comparison you can tune
N_case = 16
TARGET_CASE = 0
SHOTS_CASE = 1000

hamover_case = run_case(
    "hamover_gap_power_classiq",
    N=N_case,
    target=TARGET_CASE,
    mode="hamover",
    approach="adiabatic",
    backend="classiq",
    schedule="gap_power",
    p=2.0,
    epsilon=0.5,
    num_qdrift=20,
    n_segments=60,
    shots=SHOTS_CASE,
)

grover_case = run_case(
    "grover_oracle_diffuser_classiq",
    N=N_case,
    target=TARGET_CASE,
    mode="grover_oracle_diffuser",
    shots=SHOTS_CASE,
    # grover_reps=None -> uses near-optimal floor(pi/4*sqrt(N))
)

single_comparison = compare_records([hamover_case, grover_case])
single_comparison

,name,N,target,algorithm,approach,backend,success_probability,valid_shot_rate,leakage_rate,found_shots,...,circuit_depth,cx_count,success_per_100_cx,success_per_100_depth,model_runtime,wall_time_s,probability_per_wall_second,algorithmic_steps,reference_error_abs,scaling_exponent
0,grover_oracle_diffuser_classiq,16,0,grover_oracle_diffuser,grover_oracle_diffuser,classiq,0.963,1.0,0.0,963,...,199,126,0.764286,0.483920,3.000000,7.123614,0.135184,3,0.001681,0.5
1,hamover_gap_power_classiq,16,0,hamiltonian_evolution,adiabatic,classiq,0.608,1.0,0.0,608,...,5915,4342,0.014003,0.010279,10.890755,160.062406,0.003799,1200,0.114345,0.5


### Counter Adabatic term comparison

In [ ]:
# circuit depth comparison

In [8]:
# Optional sweep over multiple N values
N_values = [4, 8, 16, 32]
TARGET = 0
SHOTS = 1000

records = []
for n in N_values:
    records.append(
        run_case(
            f"hamover_N{n}",
            N=n,
            target=TARGET,
            mode="hamover",
            approach="adiabatic",
            backend="classiq",
            schedule="gap_power",
            p=2.0,
            epsilon=0.5,
            num_qdrift=40,
            n_segments=60,
            shots=SHOTS,
        )
    )
    records.append(
        run_case(
            f"grover_N{n}",
            N=n,
            target=TARGET,
            mode="grover_oracle_diffuser",
            shots=SHOTS,
        )
    )

sweep_comparison = compare_records(records)
sweep_comparison

ClassiqAPIError: Call to API failed with code 504
If you need further assistance, please reach out on our Community Slack channel at: https://short.classiq.io/join-slack or open a support ticket at: https://classiq-community.freshdesk.com/support/tickets/new